In [33]:
import os


In [34]:
%pwd

'E:\\Kidney_Disease_Classificatoin'

In [35]:
os.chdir(r"E:\Kidney_Disease_Classificatoin")

In [36]:
%pwd

'E:\\Kidney_Disease_Classificatoin'

In [37]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class PrepareBaseModelConfig:
    
    root_dir: Path
    base_model_path: Path
    updated_base_model_path: Path
    params_image_size: list
    params_learning_rate: float
    params_include_top: bool
    params_weights: str
    params_weights: str
    params_classes: int

In [38]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories



In [44]:
class ConfigurationManager:
    
    def __init__(self,
                 config_filepath = CONFIG_FILE_PATH,
                 params_filepath = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])

    def get_prepare_base_model_config(self) -> PrepareBaseModelConfig:
        config = self.config.prepare_base_model
        params = self.params
        prepare_base_model_config = PrepareBaseModelConfig(
            root_dir=Path(config.root_dir),
            base_model_path=Path(config.base_model_path),
            updated_base_model_path=Path(config.updated_base_model_path),
            params_image_size=params.IMAGE_SIZE,
            params_learning_rate=params.LEARNING_RATE,
            params_include_top=params.INCLUDE_TOP,
            params_weights=params.WEIGHTS,
            params_classes=params.CLASSES
        )
        return prepare_base_model_config

In [45]:
import os
import urllib.request as request
from zipfile import ZipFile
import tensorflow as tf


In [46]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories
from cnnClassifier import logger

class prepareBaseModel:

    def __init__(self, config: PrepareBaseModelConfig):
        self.config = config

    def get_base_model(self):
        try:
            self.model = tf.keras.applications.vgg16.VGG16(
                input_shape=self.config.params_image_size,
                include_top=self.config.params_include_top,
                weights=self.config.params_weights,
                classes=self.config.params_classes
            )
            self.model.save(self.config.base_model_path)
            logger.info(f"base model is saved at {self.config.base_model_path} successfully")
        except Exception as e:
            raise e

    @staticmethod
    def prepare_full_model(model, classes, freeze_all, freeze_till, learning_rate):
        if freeze_all:
            for layer in model.layers:
                layer.trainable = False
        elif (freeze_till is not None) and (freeze_till > 0):
            for layer in model.layers[:-freeze_till]:
                layer.trainable = False

        flatten_in = tf.keras.layers.Flatten()(model.output)
        prediction = tf.keras.layers.Dense(units=classes, activation='softmax')(flatten_in)

        full_model = tf.keras.models.Model(inputs=model.input, outputs=prediction)

        full_model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
            loss='categorical_crossentropy',
            metrics=['accuracy']
        )
        return full_model

    def update_base_model(self):
        self.full_model = self.prepare_full_model(
            model=self.model,
            classes=self.config.params_classes,
            freeze_all=True,
            freeze_till=None,
            learning_rate=self.config.params_learning_rate
        )
        self.full_model.save(self.config.updated_base_model_path)
        logger.info(f"updated base model is saved at {self.config.updated_base_model_path} successfully")

In [47]:
try:
    config = ConfigurationManager()
    prepare_base_model_config = config.get_prepare_base_model_config()
    prepare_base_model = prepareBaseModel(config=prepare_base_model_config)
    prepare_base_model.get_base_model()
    prepare_base_model.update_base_model()
except Exception as e:
    raise e

[2026-10-09 16:43:15,803: INFO: common]: yaml file: config\config.yaml loaded successfully
[2026-10-09 16:43:15,806: INFO: common]: yaml file: params.yaml loaded successfully
[2026-10-09 16:43:15,808: INFO: common]: created directory at: artifacts
[2026-10-09 16:43:16,154: WARNING: saving_utils]: Compiled the loaded model, but the compiled metrics have yet to be built. `model.compile_metrics` will be empty until you train or evaluate the model.
[2026-10-09 16:43:16,232: INFO: 1118231086]: base model is saved at artifacts\prepare_base_model\base_model.h5 successfully
[2026-10-09 16:43:16,363: INFO: 1118231086]: updated base model is saved at artifacts\prepare_base_model\base_model_updated.h5 successfully


In [ ]:
%pwd 

'e:\\'

AttributeError: module 'tensorflow' has no attribute '__version__'